# Bland-Altman Standalone Performance Analysis
### Bland-Altman Standalone Performans Analizi

**EN:** This notebook compares an AI measurement with the mean of 3 expert readers, for two synthetic studies: **AI bladder volume (mL)** and **AI liver craniocaudal length (cm)**. It shows every step, from the raw differences to the PASS/FAIL decision.

**TR:** Bu notebook bir AI ölçümünü 3 uzman okuyucunun ortalamasıyla karşılaştırır. İki sentetik çalışma var: **AI mesane hacmi (mL)** ve **AI karaciğer kraniokaudal uzunluğu (cm)**. Ham farklardan PASS/FAIL kararına kadar her adımı gösterir.

> ⚠️ **EN:** The data are **synthetic**, for training only. They are not real patient data and do not represent any device.
> **TR:** Veriler **sentetiktir**, yalnızca eğitim amaçlıdır. Gerçek hasta verisi değildir ve hiçbir cihazı temsil etmez.

| Step | EN | TR |
|---|---|---|
| 0 | Setup and configuration | Kurulum ve ayarlar |
| 1 | Data loading and patient flow | Veri yükleme ve hasta akışı |
| 2 | Primary endpoint, step by step | Birincil sonlanım, adım adım |
| 3 | Why correlation is not enough | Neden korelasyon yetmez |
| 4 | Secondary endpoint and choice of scale | İkincil sonlanım ve ölçek seçimi |
| 5 | ICC, MAE and clinical cut-off | ICC, MAE ve klinik eşik |
| 6 | AI vs. human readers | AI ve insan okuyucular |
| 7 | Subgroups | Alt gruplar |
| 8 | Sample size planning | Örneklem büyüklüğü planlama |
| 9 | Save outputs | Çıktıları kaydet |
| 10 | How to run | Nasıl çalıştırılır |

A companion guide, `bland_altman_guide.md`, explains the method in more depth.
Yöntemi daha ayrıntılı anlatan `bland_altman_guide.md` rehberi de var.

## 0. Setup and configuration / Kurulum ve ayarlar

**EN:** Set `STUDY` to `"bladder"` or `"liver"`, then run all cells (*Kernel → Restart & Run All*). If the CSV files are not in the same folder, the notebook generates them with a fixed seed.

**TR:** `STUDY` değişkenini `"bladder"` veya `"liver"` yap, sonra tüm hücreleri çalıştır (*Kernel → Restart & Run All*). CSV dosyaları aynı klasörde değilse notebook onları sabit bir seed ile üretir.

In [2]:
# ===== USER SETTINGS =====
STUDY = "liver"          # "bladder" or "liver"
CSV_PATH = 'liver_study.csv'            # None -> bladder_study.csv / liver_study.csv
RUN_POWER = True           # sample size simulation (~1 min)
SAVE_OUTPUTS = True        # save figures and tables to results_<study>/
# =========================

import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

%matplotlib inline
pd.set_option("display.precision", 2)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

COL = {"ink": "#14213D", "accent": "#C2621B", "blue": "#2B6CB0", "grey": "#8A94A6",
       "band": "#E9EDF3", "fail": "#C53030", "pass": "#2F855A"}

### Acceptance limits and rationale / Kabul sınırları ve gerekçeleri

| Study | Primary endpoint | Limit | Rationale (EN) | Gerekçe (TR) |
|---|---|---|---|---|
| Bladder | Absolute difference (mL) | **±55 mL** | Inter-sonographer agreement for manual 2D US was 3.2 ± 54.4 mL (PMC11353831, 2024). FDA K230497 Bladder AI reported bias 2 mL, LoA −42/+46 mL, which falls inside this limit. | 2D US'de iki sonografer arası uyum 3.2 ± 54.4 mL (PMC11353831, 2024). FDA K230497 Bladder AI: bias 2 mL, LoA −42/+46 mL; bu sınırın içinde. |
| Liver | Percent difference (%) | **±15 %** | No published numeric LoA was found for liver length, so FDA organ-length precedents were used: K260667 kidney ±14.3 %, spleen ±15.9 %. Liver inter-rater ICC is 0.71–0.94 (Childs 2014). | Karaciğer uzunluğu için yayımlanmış sayısal LoA bulunamadı; FDA organ uzunluğu emsalleri kullanıldı: K260667 böbrek ±%14.3, dalak ±%15.9. Karaciğer okuyucular arası ICC 0.71–0.94 (Childs 2014). |

**Clinical cut-offs / Klinik eşikler:** PVR > 200 mL; hepatomegaly / hepatomegali > 16 cm.

**EN:** In a real study the limit is written into the protocol before unblinding and confirmed with FDA in a Pre-Sub.
**TR:** Gerçek bir çalışmada sınır, körlük kaldırılmadan önce protokole yazılır ve FDA ile Pre-Sub görüşmesinde teyit edilir.

In [3]:
CONFIG = {
    "bladder": {
        "title": "AI Bladder Volume",
        "unit": "mL",
        "readers": ["reader1_ml", "reader2_ml", "reader3_ml"],
        "ai": "ai_ml",
        "primary": {"name": "Absolute difference (mL)", "scale": "abs", "limit": 55.0},
        "secondary": {"name": "Percent difference, reference >= 100 mL (%)", "scale": "pct",
                      "limit": None, "min_ref": 100.0},
        "cutoff": {"value": 200.0, "label": "PVR > 200 mL (elevated post-void residual)"},
        "power": {"bias": 2.0, "sd": 22.0, "delta": 55.0},
    },
    "liver": {
        "title": "AI Liver Craniocaudal Length (MCL)",
        "unit": "cm",
        "readers": ["reader1_cm", "reader2_cm", "reader3_cm"],
        "ai": "ai_cm",
        "primary": {"name": "Percent difference (%)", "scale": "pct", "limit": 15.0},
        "secondary": {"name": "Absolute difference (cm)", "scale": "abs", "limit": None, "min_ref": None},
        "cutoff": {"value": 16.0, "label": "Hepatomegaly > 16 cm"},
        "power": {"bias": -1.5, "sd": 5.0, "delta": 15.0},
    },
}
cfg = CONFIG[STUDY]
U = cfg["unit"]
OUT = f"results_{STUDY}"
if SAVE_OUTPUTS:
    os.makedirs(OUT, exist_ok=True)

def scale_unit(scale):
    return "%" if scale == "pct" else U

print(f"Study: {cfg['title']}")
print(f"Primary endpoint: {cfg['primary']['name']}, limit ±{cfg['primary']['limit']:g} {scale_unit(cfg['primary']['scale'])}")

Study: AI Liver Craniocaudal Length (MCL)
Primary endpoint: Percent difference (%), limit ±15 %


### Synthetic data generator / Sentetik veri üretici

**EN:** Runs only if the CSV files are missing. It simulates 3 sites, cart-based and handheld devices, 3 readers with small systematic differences (reader 3 measures slightly larger), and a few AI outliers, so the plots show realistic patterns.

**TR:** Yalnızca CSV dosyaları yoksa çalışır. 3 merkez, cart-based ve handheld cihazlar, küçük sistematik farkları olan 3 okuyucu (okuyucu 3 biraz büyük ölçüyor) ve birkaç AI aykırı değeri simüle eder; böylece grafiklerde gerçekçi desenler görülür.

In [4]:
SEED = 2026

def _demographics(rng, n):
    age = np.clip(rng.normal(55, 17, n), 18, 92).round().astype(int)
    sex = rng.choice(["F", "M"], n, p=[0.52, 0.48])
    bmi = np.clip(rng.lognormal(np.log(27), 0.18, n), 17, 48).round(1)
    bmi_cat = pd.cut(bmi, [0, 25, 30, 100], labels=["<25", "25-30", ">=30"]).astype(str)
    site = rng.choice(["Site_A", "Site_B", "Site_C"], n, p=[0.40, 0.35, 0.25])
    device = rng.choice(["Cart-based", "Handheld"], n, p=[0.6, 0.4])
    return age, sex, bmi, bmi_cat, site, device

def make_bladder(rng, n=150):
    age, sex, bmi, bmi_cat, site, device = _demographics(rng, n)
    true_v = np.clip(rng.lognormal(np.log(170), 0.75, n), 15, 650)      # true volume
    obese, handheld = bmi >= 30, device == "Handheld"
    readers = {}
    for name, b in {"reader1_ml": -0.02, "reader2_ml": 0.00, "reader3_ml": 0.025}.items():
        sd = (3 + 0.07 * true_v) * np.where(obese, 1.15, 1.0)
        readers[name] = np.clip(true_v * (1 + b) + rng.normal(0, sd), 5, None)
    ai_sd = (2.5 + 0.045 * true_v) * np.where(obese, 1.25, 1.0) * np.where(handheld, 1.15, 1.0)
    ai = true_v * 1.015 + 2 + rng.normal(0, ai_sd)
    out_idx = rng.choice(n, 2, replace=False)
    ai[out_idx] += np.array([70, -55])                                   # segmentation errors
    ai = np.clip(ai, 5, None)
    quality = rng.choice(["Good", "Adequate", "Poor"], n, p=[0.62, 0.32, 0.06])
    excl = np.full(n, "", dtype=object)
    excl[quality == "Poor"] = "Poor image quality"
    excl[rng.choice(np.where(quality != "Poor")[0], 3, replace=False)] = "Bladder not fully in field of view"
    df = pd.DataFrame({"case_id": [f"BLD-{i+1:03d}" for i in range(n)], "site": site,
                       "device_type": device, "age": age, "sex": sex, "bmi": bmi,
                       "bmi_category": bmi_cat, "image_quality": quality,
                       "analyzable": np.where(excl == "", "Yes", "No"), "exclusion_reason": excl,
                       **{k: v.round(1) for k, v in readers.items()}, "ai_ml": ai.round(1)})
    df.loc[df.analyzable == "No", list(readers) + ["ai_ml"]] = np.nan
    return df

def make_liver(rng, n=160):
    age, sex, bmi, bmi_cat, site, device = _demographics(rng, n)
    true_l = np.clip(rng.normal(15.3, 2.3, n) + np.where(bmi >= 30, 0.8, 0), 10, 23)
    obese = bmi >= 30
    readers = {}
    for name, b in {"reader1_cm": -0.02, "reader2_cm": 0.0, "reader3_cm": 0.02}.items():
        readers[name] = true_l * (1 + b) + rng.normal(0, 0.052 * true_l * np.where(obese, 1.2, 1.0))
    ai = true_l * 0.985 + rng.normal(0, 0.038 * true_l * np.where(obese, 1.3, 1.0))
    ai[rng.choice(n, 1)] -= 3.0                                          # missed Riedel lobe
    quality = rng.choice(["Good", "Adequate", "Poor"], n, p=[0.58, 0.35, 0.07])
    excl = np.full(n, "", dtype=object)
    excl[quality == "Poor"] = "Poor image quality / rib shadowing"
    excl[rng.choice(np.where(quality != "Poor")[0], 3, replace=False)] = "Liver tip not visualized"
    df = pd.DataFrame({"case_id": [f"LVR-{i+1:03d}" for i in range(n)], "site": site,
                       "device_type": device, "age": age, "sex": sex, "bmi": bmi,
                       "bmi_category": bmi_cat, "image_quality": quality,
                       "analyzable": np.where(excl == "", "Yes", "No"), "exclusion_reason": excl,
                       **{k: v.round(2) for k, v in readers.items()}, "ai_cm": ai.round(2)})
    df.loc[df.analyzable == "No", list(readers) + ["ai_cm"]] = np.nan
    return df

def ensure_csvs():
    # Same RNG order as the original script -> identical CSVs
    if not (os.path.exists("bladder_study.csv") and os.path.exists("liver_study.csv")):
        rng = np.random.default_rng(SEED)
        make_bladder(rng).to_csv("bladder_study.csv", index=False)
        make_liver(rng).to_csv("liver_study.csv", index=False)
        print("CSV files generated.")

if CSV_PATH is None:
    ensure_csvs()
    CSV_PATH = f"{STUDY}_study.csv"
print("Data file:", CSV_PATH)

Data file: liver_study.csv


### Statistical and plotting functions / İstatistik ve grafik fonksiyonları

| Function | EN | TR |
|---|---|---|
| `differences` | `abs`: AI − ref; `pct`: (AI − ref) / mean × 100 | `abs`: AI − ref; `pct`: (AI − ref) / ortalama × 100 |
| `bland_altman` | Bias, SD, 95 % LoA and their confidence intervals (Bland & Altman 1999) | Bias, SD, %95 LoA ve güven aralıkları (Bland & Altman 1999) |
| `acceptance` | *point*: LoA inside the limit; *strict*: LoA CIs also inside | *nokta*: LoA sınır içinde; *sıkı*: LoA güven aralıkları da sınır içinde |
| `proportional_bias` | Regression of the difference on the mean | Farkın ortalamaya göre regresyonu |
| `icc_a1` | ICC(A,1), absolute agreement | ICC(A,1), mutlak uyum |
| `categorical` | Sensitivity, specificity, kappa at a clinical cut-off | Klinik eşikte duyarlılık, özgüllük, kappa |

In [5]:
def differences(new, ref, scale):
    # 'abs' = new - ref ; 'pct' = (new - ref) / mean(new, ref) * 100
    new, ref = np.asarray(new, float), np.asarray(ref, float)
    return new - ref if scale == "abs" else (new - ref) / ((new + ref) / 2) * 100

def bland_altman(diff, alpha=0.05):
    # SE(LoA) ~= SD * sqrt(1/n + z^2 / (2(n-1)))   (Bland & Altman 1999)
    d = np.asarray(diff, float); d = d[~np.isnan(d)]
    n, bias, sd = len(d), d.mean(), d.std(ddof=1)
    z, t = stats.norm.ppf(0.975), stats.t.ppf(1 - alpha / 2, n - 1)
    se_b, se_l = sd / np.sqrt(n), sd * np.sqrt(1 / n + z**2 / (2 * (n - 1)))
    lo, hi = bias - z * sd, bias + z * sd
    return {"n": n, "bias": bias, "sd": sd, "z": z, "t": t, "se_bias": se_b, "se_loa": se_l,
            "bias_ci_lo": bias - t * se_b, "bias_ci_hi": bias + t * se_b,
            "loa_lo": lo, "loa_hi": hi,
            "loa_lo_ci_lo": lo - t * se_l, "loa_lo_ci_hi": lo + t * se_l,
            "loa_hi_ci_lo": hi - t * se_l, "loa_hi_ci_hi": hi + t * se_l}

def acceptance(res, limit):
    if limit is None:
        return {"point": None, "strict": None}
    return {"point": res["loa_lo"] >= -limit and res["loa_hi"] <= limit,
            "strict": res["loa_lo_ci_lo"] >= -limit and res["loa_hi_ci_hi"] <= limit}

def verdict(acc):
    if acc["point"] is None:
        return "No pre-specified limit (descriptive)"
    s = lambda v: "PASS" if v else "FAIL"
    return f"LoA point estimate: {s(acc['point'])} | LoA 95% CI (strict): {s(acc['strict'])}"

def proportional_bias(new, ref, diff):
    mean = (np.asarray(new) + np.asarray(ref)) / 2
    r = stats.linregress(mean, diff)
    return {"slope": r.slope, "intercept": r.intercept, "p": r.pvalue}

def icc_a1(x, y):
    # ICC(A,1): two-way random effects, absolute agreement, single measure (McGraw & Wong 1996)
    data = np.column_stack([x, y]); n, k = data.shape; gm = data.mean()
    ms_r = k * ((data.mean(1) - gm) ** 2).sum() / (n - 1)
    ms_c = n * ((data.mean(0) - gm) ** 2).sum() / (k - 1)
    ss_e = ((data - data.mean(1, keepdims=True) - data.mean(0, keepdims=True) + gm) ** 2).sum()
    ms_e = ss_e / ((n - 1) * (k - 1))
    return (ms_r - ms_e) / (ms_r + (k - 1) * ms_e + k * (ms_c - ms_e) / n)

def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return p, c - h, c + h

def categorical(ai, ref, cutoff):
    a, r = np.asarray(ai) > cutoff, np.asarray(ref) > cutoff
    tp, tn, fp, fn = int((a & r).sum()), int((~a & ~r).sum()), int((a & ~r).sum()), int((~a & r).sum())
    n = tp + tn + fp + fn; po = (tp + tn) / n
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / n**2
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "sens": wilson(tp, tp + fn),
            "spec": wilson(tn, tn + fp), "agreement": po, "kappa": (po - pe) / (1 - pe)}

def res_table(res):
    return pd.DataFrame([{
        "n": res["n"], "Bias": res["bias"],
        "Bias 95% CI": f"{res['bias_ci_lo']:+.2f} / {res['bias_ci_hi']:+.2f}",
        "SD": res["sd"], "Lower LoA": res["loa_lo"],
        "Lower LoA 95% CI": f"{res['loa_lo_ci_lo']:+.2f} / {res['loa_lo_ci_hi']:+.2f}",
        "Upper LoA": res["loa_hi"],
        "Upper LoA 95% CI": f"{res['loa_hi_ci_lo']:+.2f} / {res['loa_hi_ci_hi']:+.2f}"}])

def save(fig, name):
    if SAVE_OUTPUTS:
        fig.savefig(os.path.join(OUT, name), dpi=160, bbox_inches="tight")

def ylabel_for(scale):
    return ("Difference: (AI - Ref) / mean x 100 (%)" if scale == "pct"
            else f"Difference: AI - Reference ({U})")

def plot_ba(new, ref, scale, res, limit, title, fname=None, highlight=None, ax=None, legend=True):
    mean = (np.asarray(new) + np.asarray(ref)) / 2
    diff = differences(new, ref, scale)
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(11, 6.5))
    if limit is not None:
        ax.axhspan(-limit, limit, color=COL["band"], zorder=0, label=f"Acceptance limit ±{limit:g}")
    ax.scatter(mean, diff, s=28, color=COL["ink"], alpha=0.75, zorder=3)
    if highlight is not None and highlight.any():
        ax.scatter(mean[highlight], diff[highlight], s=90, facecolors="none", edgecolors=COL["fail"],
                   linewidths=1.6, zorder=4, label="Outside LoA")
    ax.axhline(0, color=COL["grey"], lw=1, ls=":")
    ax.axhline(res["bias"], color=COL["blue"], lw=2.2, label=f"Bias {res['bias']:+.2f}")
    ax.axhspan(res["bias_ci_lo"], res["bias_ci_hi"], color=COL["blue"], alpha=0.12)
    for key, lab in [("loa_lo", "Lower LoA"), ("loa_hi", "Upper LoA")]:
        ax.axhline(res[key], color=COL["accent"], lw=2, ls="--", label=f"{lab} {res[key]:+.2f}")
        ax.axhspan(res[key + "_ci_lo"], res[key + "_ci_hi"], color=COL["accent"], alpha=0.12)
    pb = proportional_bias(new, ref, diff)
    xs = np.linspace(mean.min(), mean.max(), 50)
    ax.plot(xs, pb["intercept"] + pb["slope"] * xs, color=COL["ink"], lw=1, ls="-.", alpha=0.6,
            label=f"Trend (slope p={pb['p']:.3f})")
    ax.set_title(title, loc="left", fontsize=11, color=COL["ink"])
    ax.set_xlabel(f"Mean of AI and reference ({U})")
    ax.set_ylabel(ylabel_for(scale))
    ax.grid(alpha=0.25)
    if legend:
        ax.legend(fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    if own:
        fig.tight_layout()
        if fname:
            save(fig, fname)
        plt.show()

def plot_intervals(rows, limit, title, xlabel, fname):
    fig, ax = plt.subplots(figsize=(10, 0.5 * len(rows) + 1.8))
    y = np.arange(len(rows))[::-1]
    if limit is not None:
        ax.axvspan(-limit, limit, color=COL["band"], zorder=0, label=f"Limit ±{limit:g}")
    for yi, (_, r) in zip(y, rows.iterrows()):
        c = COL["accent"] if r.get("is_ai", False) else COL["ink"]
        ax.plot([r["loa_lo"], r["loa_hi"]], [yi, yi], color=c, lw=3, solid_capstyle="butt")
        ax.plot(r["bias"], yi, "o", color=COL["blue"], ms=7, zorder=3)
        ax.text(r["loa_hi"], yi + 0.18, f" n={int(r['n'])}", fontsize=8, color=COL["grey"])
    ax.axvline(0, color=COL["grey"], lw=1, ls=":")
    ax.set_yticks(y); ax.set_yticklabels(rows["label"], fontsize=9)
    ax.set_xlabel(xlabel); ax.set_title(title, loc="left", fontsize=12); ax.grid(alpha=0.25)
    if limit is not None:
        ax.legend(fontsize=9, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout(); save(fig, fname); plt.show()

## 1. Data loading and patient flow / Veri yükleme ve hasta akışı

**EN:** Cases that cannot be analysed (poor image quality, organ not fully visible) are excluded and reported, as in a STARD flow diagram. The **reference standard** for each case is the mean of the 3 readers.

**TR:** Analiz edilemeyen vakalar (kötü görüntü kalitesi, organın tamamen görünmemesi) dışlanır ve STARD akış şemasındaki gibi raporlanır. Her vaka için **referans standart** 3 okuyucunun ortalamasıdır.

In [6]:
raw = pd.read_csv(CSV_PATH)
excl = raw[raw["analyzable"] != "Yes"]
df = raw[raw["analyzable"] == "Yes"].copy()
df["reference"] = df[cfg["readers"]].mean(axis=1)
df["age_group"] = pd.cut(df["age"], [0, 39, 64, 200], labels=["18-39", "40-64", "65+"]).astype(str)
ai, ref = df[cfg["ai"]].to_numpy(), df["reference"].to_numpy()

print(f"Enrolled: {len(raw)}   Excluded: {len(excl)}   Analyzed: {len(df)}")
display(excl["exclusion_reason"].value_counts().rename("n").to_frame().rename_axis("Exclusion reason"))
print(f"Reference range: {ref.min():.1f} - {ref.max():.1f} {U} (median {np.median(ref):.1f})")
display(df.head())

Enrolled: 160   Excluded: 13   Analyzed: 147


,n
Exclusion reason,
Poor image quality / rib shadowing,10
Liver tip not visualized,3


Reference range: 10.3 - 20.0 cm (median 15.2)


,case_id,site,device_type,age,sex,bmi,bmi_category,image_quality,analyzable,exclusion_reason,reader1_cm,reader2_cm,reader3_cm,ai_cm,reference,age_group
0,LVR-001,Site_B,Cart-based,65,M,23.2,<25,Good,Yes,NaN,16.29,16.65,17.24,15.80,16.73,65+
1,LVR-002,Site_C,Cart-based,50,F,23.2,<25,Adequate,Yes,NaN,17.58,15.87,16.62,16.02,16.69,40-64
2,LVR-003,Site_A,Cart-based,52,F,27.6,25-30,Good,Yes,NaN,11.48,11.01,10.68,10.53,11.06,40-64
3,LVR-004,Site_A,Handheld,59,F,32.5,>=30,Good,Yes,NaN,13.17,14.60,16.84,13.61,14.87,40-64
4,LVR-005,Site_A,Cart-based,42,M,30.5,>=30,Adequate,Yes,NaN,15.07,16.23,16.62,15.91,15.97,40-64


## 2. Primary endpoint, step by step / Birincil sonlanım, adım adım

**EN:** The question is: *for an individual patient, how far can the AI measurement be from the expert reference?* Bland-Altman answers this in five steps:

1. Compute the **difference** for each patient.
2. Average the differences: the **bias** (systematic shift).
3. Compute the **SD of the differences** (how much the shift varies between patients).
4. Build the **95 % limits of agreement (LoA)**: bias ± 1.96 × SD.
5. Compare the LoA, and their confidence intervals, with the **pre-specified acceptance limit**.

Then two assumptions are checked (no proportional error, approximately normal differences) and the result is drawn as the Bland-Altman plot.

**TR:** Soru şu: *tek bir hastada AI ölçümü uzman referansından ne kadar uzak olabilir?* Bland-Altman bunu beş adımda cevaplar:

1. Her hasta için **farkı** hesapla.
2. Farkların ortalamasını al: **bias** (sistematik sapma).
3. **Farkların SD'sini** hesapla (sapmanın hastadan hastaya ne kadar değiştiği).
4. **%95 uyum sınırlarını (LoA)** oluştur: bias ± 1.96 × SD.
5. LoA'yı ve güven aralıklarını **önceden belirlenen kabul sınırıyla** karşılaştır.

Ardından iki varsayım kontrol edilir (orantısal hata yok, farklar yaklaşık normal) ve sonuç Bland-Altman grafiğine dökülür.

### 2.1 Differences for each patient / Her hasta için fark

**EN:** The table below shows the first 8 patients. `reference` is the mean of the 3 readers, and `difference` is the AI value minus the reference on the primary scale. The sign matters: **positive means the AI measures larger** than the experts, negative means smaller.

- **Absolute scale (`abs`):** difference = AI − reference, in the measurement unit. Used when the error is roughly the same size across the measuring range.
- **Percent scale (`pct`):** difference = (AI − reference) / mean(AI, reference) × 100. Used when the error grows with the size of the organ, so a fixed number of mL or cm would be misleading.

**TR:** Aşağıdaki tablo ilk 8 hastayı gösteriyor. `reference` 3 okuyucunun ortalaması, `difference` ise birincil ölçekte AI değeri eksi referans. İşaret önemlidir: **pozitif değer AI'ın uzmanlardan büyük ölçtüğünü**, negatif değer küçük ölçtüğünü gösterir.

- **Mutlak ölçek (`abs`):** fark = AI − referans, ölçüm biriminde. Hata ölçüm aralığı boyunca yaklaşık aynı büyüklükteyse kullanılır.
- **Yüzde ölçek (`pct`):** fark = (AI − referans) / ortalama(AI, referans) × 100. Hata organın büyüklüğüyle birlikte büyüyorsa kullanılır; o durumda sabit bir mL ya da cm değeri yanıltıcı olur.

In [7]:
pr = cfg["primary"]
PU = scale_unit(pr["scale"])
d1 = differences(ai, ref, pr["scale"])

example = df[["case_id"] + cfg["readers"] + ["reference", cfg["ai"]]].head(8).copy()
example["mean(AI, ref)"] = (example[cfg["ai"]] + example["reference"]) / 2
example[f"difference ({PU})"] = d1[:8]
display(example)
print(f"Primary scale: {pr['scale']}  ->  difference is expressed in {PU}")
print(f"All {len(d1)} differences: min {d1.min():+.2f}, median {np.median(d1):+.2f}, max {d1.max():+.2f} {PU}")

,case_id,reader1_cm,reader2_cm,reader3_cm,reference,ai_cm,"mean(AI, ref)",difference (%)
0,LVR-001,16.29,16.65,17.24,16.73,15.80,16.26,-5.70
1,LVR-002,17.58,15.87,16.62,16.69,16.02,16.35,-4.10
2,LVR-003,11.48,11.01,10.68,11.06,10.53,10.79,-4.88
3,LVR-004,13.17,14.60,16.84,14.87,13.61,14.24,-8.85
4,LVR-005,15.07,16.23,16.62,15.97,15.91,15.94,-0.40
5,LVR-006,14.90,17.40,15.79,16.03,15.67,15.85,-2.27
6,LVR-007,15.22,16.71,17.24,16.39,16.84,16.62,2.71
8,LVR-009,13.80,13.91,15.63,14.45,11.08,12.76,-26.38


Primary scale: pct  ->  difference is expressed in %
All 147 differences: min -26.38, median -2.27, max +11.71 %


### 2.2 Bias, SD and limits of agreement / Bias, SD ve uyum sınırları

**EN:**
- **Bias** $= \bar d = \frac{1}{n}\sum d_i$. It is the average systematic shift. A bias of 0 means the AI neither over- nor under-measures on average. The bias alone says nothing about individual patients: +50 and −50 average to 0.
- **SD of the differences** $= s_d$. It measures how scattered the individual differences are around the bias. This is the random part of the disagreement.
- **95 % limits of agreement** $= \bar d \pm 1.96\, s_d$. If the differences are roughly normal, about 95 % of patients will have a difference between the lower and upper LoA. **This is the clinically useful statement:** "for 95 % of patients, the AI differs from the experts by between X and Y."

The value 1.96 is the number of standard deviations that covers the central 95 % of a normal distribution.

**TR:**
- **Bias** $= \bar d = \frac{1}{n}\sum d_i$. Ortalama sistematik sapmadır. Bias 0 ise AI ortalamada ne fazla ne eksik ölçer. Bias tek başına bireysel hastalar hakkında bir şey söylemez: +50 ile −50'nin ortalaması da 0'dır.
- **Farkların SD'si** $= s_d$. Bireysel farkların bias etrafında ne kadar saçıldığını ölçer. Uyumsuzluğun rastgele kısmıdır.
- **%95 uyum sınırları** $= \bar d \pm 1.96\, s_d$. Farklar yaklaşık normal dağılıyorsa hastaların yaklaşık %95'inde fark alt ve üst LoA arasında olur. **Klinik olarak kullanışlı ifade budur:** "Hastaların %95'inde AI uzmanlardan X ile Y arasında farklı ölçer."

1.96, normal dağılımın ortadaki %95'ini kapsayan standart sapma sayısıdır.

In [9]:
r1 = bland_altman(d1)
print(f"n            = {r1['n']}")
print(f"Bias         = sum(d) / n                = {d1.sum():.2f} / {r1['n']} = {r1['bias']:+.2f} {PU}")
print(f"SD(d)        = {r1['sd']:.2f} {PU}")
print(f"Lower LoA    = bias - 1.96 x SD          = {r1['bias']:+.2f} - 1.96 x {r1['sd']:.2f} = {r1['loa_lo']:+.2f} {PU}")
print(f"Upper LoA    = bias + 1.96 x SD          = {r1['bias']:+.2f} + 1.96 x {r1['sd']:.2f} = {r1['loa_hi']:+.2f} {PU}")
print(f"LoA width    = {r1['loa_hi'] - r1['loa_lo']:.2f} {PU}")

n            = 147
Bias         = sum(d) / n                = -312.90 / 147 = -2.13 %
SD(d)        = 5.60 %
Lower LoA    = bias - 1.96 x SD          = -2.13 - 1.96 x 5.60 = -13.10 %
Upper LoA    = bias + 1.96 x SD          = -2.13 + 1.96 x 5.60 = +8.84 %
LoA width    = 21.94 %


In [10]:
display(Markdown(
f"**EN:** On average the AI measures **{abs(r1['bias']):.2f} {PU} {'larger' if r1['bias'] > 0 else 'smaller'}** than the expert reference. "
f"For about 95 % of patients, the AI result will lie between **{r1['loa_lo']:+.2f} and {r1['loa_hi']:+.2f} {PU}** of the reference.\n\n"
f"**TR:** AI ortalamada uzman referansından **{abs(r1['bias']):.2f} {PU} {'büyük' if r1['bias'] > 0 else 'küçük'}** ölçüyor. "
f"Hastaların yaklaşık %95'inde AI sonucu referanstan **{r1['loa_lo']:+.2f} ile {r1['loa_hi']:+.2f} {PU}** arasında farklı olacak."))

**EN:** On average the AI measures **2.13 % smaller** than the expert reference. For about 95 % of patients, the AI result will lie between **-13.10 and +8.84 %** of the reference.

**TR:** AI ortalamada uzman referansından **2.13 % küçük** ölçüyor. Hastaların yaklaşık %95'inde AI sonucu referanstan **-13.10 ile +8.84 %** arasında farklı olacak.

### 2.3 Confidence intervals of the bias and the LoA / Bias ve LoA'nın güven aralıkları

**EN:** The bias and the LoA are estimated from a sample, so they carry uncertainty. With a different set of patients they would come out slightly different. Their 95 % confidence intervals (CIs) show how precisely they are estimated:

- $SE(\bar d) = s_d / \sqrt{n}$, so the CI of the bias is $\bar d \pm t \cdot SE(\bar d)$.
- $SE(\text{LoA}) \approx s_d \sqrt{\frac{1}{n} + \frac{1.96^2}{2(n-1)}}$, so each LoA gets its own CI (Bland & Altman 1999).

**Why it matters:** if the CI of the bias does not contain 0, the systematic shift is statistically real. The CI of the LoA is what the *strict* acceptance criterion uses. A larger n gives narrower CIs, which is the basis of the sample size calculation in step 8.

**TR:** Bias ve LoA bir örneklemden tahmin edilir, dolayısıyla belirsizlik taşır. Farklı bir hasta grubuyla biraz farklı çıkarlardı. %95 güven aralıkları (GA) ne kadar kesin tahmin edildiklerini gösterir:

- $SE(\bar d) = s_d / \sqrt{n}$; bias'ın GA'sı $\bar d \pm t \cdot SE(\bar d)$.
- $SE(\text{LoA}) \approx s_d \sqrt{\frac{1}{n} + \frac{1.96^2}{2(n-1)}}$; her LoA'nın kendi GA'sı vardır (Bland & Altman 1999).

**Neden önemli:** bias'ın GA'sı 0'ı içermiyorsa sistematik sapma istatistiksel olarak gerçektir. *Sıkı* kabul kriteri LoA'nın GA'sını kullanır. n büyüdükçe GA'lar daralır; 8. adımdaki örneklem hesabı buna dayanır.

In [11]:
print(f"t (df={r1['n']-1}) = {r1['t']:.3f}")
print(f"SE(bias) = SD / sqrt(n) = {r1['sd']:.2f} / sqrt({r1['n']}) = {r1['se_bias']:.3f}")
print(f"SE(LoA)  = SD x sqrt(1/n + 1.96^2 / (2(n-1))) = {r1['se_loa']:.3f}")
display(res_table(r1))
zero_in = r1["bias_ci_lo"] <= 0 <= r1["bias_ci_hi"]
print("Bias CI contains 0 -> no statistically significant systematic shift" if zero_in
      else "Bias CI excludes 0 -> statistically significant systematic shift")

t (df=146) = 1.976
SE(bias) = SD / sqrt(n) = 5.60 / sqrt(147) = 0.462
SE(LoA)  = SD x sqrt(1/n + 1.96^2 / (2(n-1))) = 0.791


,n,Bias,Bias 95% CI,SD,Lower LoA,Lower LoA 95% CI,Upper LoA,Upper LoA 95% CI
0,147,-2.13,-3.04 / -1.22,5.6,-13.1,-14.66 / -11.54,8.84,+7.28 / +10.41


Bias CI excludes 0 -> statistically significant systematic shift


### 2.4 Decision against the acceptance limit / Kabul sınırına göre karar

**EN:** Bland-Altman does not say by itself whether the agreement is good enough. The decision is made by comparing the LoA with the limit written in the protocol. Two levels are used:

- **Point criterion:** the lower LoA ≥ −limit and the upper LoA ≤ +limit.
- **Strict criterion:** the *outer ends of the LoA confidence intervals* are also inside the limit. This protects against passing by luck with a small sample. Philips used this in K260667.

The chart below shows the LoA (orange), their CIs (whiskers) and the limit (grey band) on one line.

**TR:** Bland-Altman uyumun yeterli olup olmadığını tek başına söylemez. Karar, LoA'yı protokolde yazılı sınırla karşılaştırarak verilir. İki seviye kullanılır:

- **Nokta kriteri:** alt LoA ≥ −sınır ve üst LoA ≤ +sınır.
- **Sıkı kriter:** *LoA güven aralıklarının dış uçları* da sınırın içinde. Küçük örneklemde şans eseri geçmeye karşı korur. Philips K260667'de bunu kullandı.

Aşağıdaki grafik LoA'yı (turuncu), güven aralıklarını (bıyıklar) ve sınırı (gri bant) tek çizgide gösteriyor.

In [ ]:
a1 = acceptance(r1, pr["limit"])
L = pr["limit"]
fig, ax = plt.subplots(figsize=(11, 2.3))
ax.axvspan(-L, L, color=COL["band"], label=f"Acceptance limit ±{L:g} {PU}")
for k in ["loa_lo", "loa_hi"]:
    ax.errorbar(r1[k], 0, xerr=[[r1[k] - r1[k + "_ci_lo"]], [r1[k + "_ci_hi"] - r1[k]]],
                fmt="o", color=COL["accent"], capsize=8, ms=9, lw=2)
    ax.text(r1[k], 0.3, f"{r1[k]:+.2f}", ha="center", color=COL["accent"])
ax.errorbar(r1["bias"], 0, xerr=[[r1["bias"] - r1["bias_ci_lo"]], [r1["bias_ci_hi"] - r1["bias"]]],
            fmt="o", color=COL["blue"], capsize=6, ms=8, lw=2, label="Bias (95% CI)")
ax.plot([], [], "o", color=COL["accent"], label="LoA (95% CI)")
ax.axvline(0, color=COL["grey"], ls=":", lw=1)
ax.set_xlim(-L * 1.35, L * 1.35); ax.set_ylim(-0.6, 0.7); ax.set_yticks([])
ax.set_xlabel(ylabel_for(pr["scale"]))
ax.set_title("Primary endpoint vs. acceptance limit", loc="left", fontsize=11)
ax.legend(fontsize=8.5, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.1))
fig.tight_layout(); save(fig, "00_decision_line.png"); plt.show()

print("DECISION:", verdict(a1))
print(f"Margin to limit: lower {r1['loa_lo_ci_lo'] + L:.2f} {PU}, upper {L - r1['loa_hi_ci_hi']:.2f} {PU} (strict, CI ends)")

### 2.5 Checking the assumptions / Varsayımların kontrolü

**EN:** The LoA are only meaningful if two assumptions hold reasonably well.

**a) No proportional error.** The difference should not grow or shrink with the size of the measurement. This is tested by regressing the difference on the mean. A significant slope means the AI's *average* error changes with size, for example it over-measures large organs. The remedy is to switch to the percent scale or to log-transform. Note that this test looks at the shift, not the spread: whether the differences *fan out* with size (heteroscedasticity) is checked separately in section 4.1.

**b) Approximately normal differences.** The 1.96 multiplier assumes normality. It is checked with a histogram, a Q-Q plot and the Shapiro-Wilk test. With heavy tails or outliers, also report **nonparametric LoA** (the 2.5th and 97.5th percentiles of the differences). Shapiro-Wilk becomes significant easily with large n, so the plots matter more than the p-value.

**c) Outliers.** About 5 % of cases are expected outside the LoA. Each one should be reviewed (segmentation failure? unusual anatomy?), but **not removed** after the fact.

**TR:** LoA ancak iki varsayım makul ölçüde sağlanıyorsa anlamlıdır.

**a) Orantısal hata yok.** Fark, ölçümün büyüklüğüyle birlikte büyümemeli ya da küçülmemeli. Bu, farkın ortalamaya göre regresyonuyla test edilir. Anlamlı bir eğim, AI'ın *ortalama* hatasının boyutla değiştiği anlamına gelir; örneğin büyük organları fazla ölçmesi. Çözüm yüzde ölçeğe geçmek ya da log dönüşümü yapmaktır. Bu test sapmaya bakar, yayılıma değil: farkların boyutla *yelpaze gibi açılıp açılmadığı* (heteroskedastisite) 4.1 bölümünde ayrıca kontrol edilir.

**b) Farklar yaklaşık normal.** 1.96 çarpanı normallik varsayar. Histogram, Q-Q grafiği ve Shapiro-Wilk testiyle kontrol edilir. Kalın kuyruk veya aykırı değer varsa ek olarak **nonparametrik LoA** (farkların 2.5. ve 97.5. persentilleri) raporlanır. n büyükken Shapiro-Wilk kolayca anlamlı çıkar; bu yüzden grafikler p değerinden daha önemlidir.

**c) Aykırı değerler.** Vakaların yaklaşık %5'inin LoA dışında kalması beklenir. Her biri incelenmeli (segmentasyon hatası mı, sıra dışı anatomi mi?), ama sonradan **çıkarılmamalıdır**.

In [ ]:
pb1 = proportional_bias(ai, ref, d1)
sw = stats.shapiro(d1)
q_np = np.percentile(d1, [2.5, 97.5])
outside = (d1 < r1["loa_lo"]) | (d1 > r1["loa_hi"])

print(f"a) Proportional error: slope = {pb1['slope']:+.4f} {PU} per {U}, p = {pb1['p']:.4f}",
      "-> significant" if pb1["p"] < 0.05 else "-> not significant")
print(f"b) Shapiro-Wilk: W = {sw.statistic:.3f}, p = {sw.pvalue:.4f}",
      "-> departure from normality" if sw.pvalue < 0.05 else "-> consistent with normality")
print(f"   Parametric LoA:    {r1['loa_lo']:+.2f} / {r1['loa_hi']:+.2f} {PU}")
print(f"   Nonparametric LoA: {q_np[0]:+.2f} / {q_np[1]:+.2f} {PU}  (2.5th / 97.5th percentile)")
print(f"c) Outside LoA: {outside.sum()} / {len(d1)} ({outside.mean()*100:.1f}%, expected ~5%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(d1, bins=25, color=COL["ink"], alpha=0.8)
for v, c in [(r1["bias"], COL["blue"]), (r1["loa_lo"], COL["accent"]), (r1["loa_hi"], COL["accent"])]:
    axes[0].axvline(v, color=c, ls="--", lw=1.6)
axes[0].set_title("Histogram of differences", loc="left", fontsize=11)
axes[0].set_xlabel(ylabel_for(pr["scale"])); axes[0].set_ylabel("Number of cases")
stats.probplot(d1, dist="norm", plot=axes[1])
axes[1].get_lines()[0].set(color=COL["ink"], markersize=4)
axes[1].get_lines()[1].set(color=COL["accent"])
axes[1].set_title("Normal Q-Q plot of differences", loc="left", fontsize=11)
for a in axes: a.grid(alpha=0.25)
fig.tight_layout(); save(fig, "00_assumption_checks.png"); plt.show()

cols = ["case_id", "site", "device_type", "bmi", "image_quality", "reference", cfg["ai"]]
out_tab = df.loc[outside, cols].copy()
out_tab[f"difference ({PU})"] = d1[outside]
print("Cases outside the LoA (review, do not remove):")
display(out_tab.sort_values(f"difference ({PU})"))

### 2.6 The Bland-Altman plot / Bland-Altman grafiği

**EN:** Each dot is one patient.
- **x-axis:** the mean of AI and reference. The true value is unknown, so this mean is the best estimate of the patient's real size.
- **y-axis:** the difference on the primary scale.
- **Blue line and band:** the bias and its 95 % CI. Distance from the dotted 0 line shows the systematic shift.
- **Orange dashed lines and bands:** the LoA and their 95 % CIs.
- **Grey band:** the acceptance limit. For a pass, the orange lines and their bands must fit inside it.
- **Dash-dot line:** the regression trend. A clear slope means proportional error.
- **Red circles:** patients outside the LoA.

Questions to ask when reading it: Is the blue line far from 0? Is the band narrow enough clinically? Do the dots fan out or slope as x increases? Are there clusters of outliers in one region?

**TR:** Her nokta bir hastadır.
- **x ekseni:** AI ve referansın ortalaması. Gerçek değer bilinmediği için hastanın gerçek boyutunun en iyi tahmini bu ortalamadır.
- **y ekseni:** birincil ölçekteki fark.
- **Mavi çizgi ve bant:** bias ve %95 GA'sı. Noktalı 0 çizgisine uzaklık sistematik sapmayı gösterir.
- **Turuncu kesikli çizgiler ve bantlar:** LoA ve %95 GA'ları.
- **Gri bant:** kabul sınırı. Geçmek için turuncu çizgilerin bantlarıyla birlikte bunun içine sığması gerekir.
- **Kesik-noktalı çizgi:** regresyon trendi. Belirgin eğim orantısal hata demektir.
- **Kırmızı halkalar:** LoA dışındaki hastalar.

Okurken sorulacak sorular: Mavi çizgi 0'dan uzak mı? Bant klinik olarak yeterince dar mı? x büyüdükçe noktalar yelpaze gibi açılıyor ya da eğim gösteriyor mu? Aykırı değerler belirli bir bölgede toplanıyor mu?

In [ ]:
plot_ba(ai, ref, pr["scale"], r1, pr["limit"],
        f"{cfg['title']}: primary Bland-Altman (n={r1['n']})",
        "01_bland_altman_primary.png", highlight=outside)

In [ ]:
trend = "a significant" if pb1["p"] < 0.05 else "no significant"
trend_tr = "anlamlı bir" if pb1["p"] < 0.05 else "anlamlı bir orantısal hata yok;"
display(Markdown(
f"**EN summary:** Bias {r1['bias']:+.2f} {PU} (95 % CI {r1['bias_ci_lo']:+.2f} to {r1['bias_ci_hi']:+.2f}); "
f"LoA {r1['loa_lo']:+.2f} to {r1['loa_hi']:+.2f} {PU} against a limit of ±{L:g} {PU}. "
f"There is {trend} proportional error (p = {pb1['p']:.3f}). {outside.sum()} of {len(d1)} cases fall outside the LoA. "
f"**{verdict(a1)}.**\n\n"
f"**TR özet:** Bias {r1['bias']:+.2f} {PU} (%95 GA {r1['bias_ci_lo']:+.2f} ile {r1['bias_ci_hi']:+.2f}); "
f"LoA {r1['loa_lo']:+.2f} ile {r1['loa_hi']:+.2f} {PU}, sınır ±{L:g} {PU}. "
+ (f"Anlamlı bir orantısal hata var (p = {pb1['p']:.3f}). " if pb1['p'] < 0.05
   else f"Anlamlı bir orantısal hata yok (p = {pb1['p']:.3f}). ")
+ f"{len(d1)} vakanın {outside.sum()} tanesi LoA dışında. "
f"Nokta kriteri: **{'GEÇTİ' if a1['point'] else 'KALDI'}**, sıkı kriter: **{'GEÇTİ' if a1['strict'] else 'KALDI'}**."))

## 3. Why correlation is not enough / Neden korelasyon yetmez

**EN:** Many method-comparison papers report only a correlation coefficient. Bland and Altman (1986) showed why this is misleading:

1. **Correlation measures association, not agreement.** r tells you whether two measurements rise together along *any* straight line. Agreement requires them to lie on *one specific* line: y = x.
2. **r ignores systematic shifts.** If the AI added a constant 30 mL to every patient, r would not change at all, even though every result would be wrong.
3. **r depends on the range of the data.** Including very small and very large organs inflates r, even when the disagreement in each patient is unchanged.

The three demonstrations below show each point on the study data.

**TR:** Birçok yöntem karşılaştırma makalesi yalnızca korelasyon katsayısı raporlar. Bland ve Altman (1986) bunun neden yanıltıcı olduğunu gösterdi:

1. **Korelasyon ilişkiyi ölçer, uyumu değil.** r, iki ölçümün *herhangi bir* doğru boyunca birlikte artıp artmadığını söyler. Uyum ise *belirli bir* doğru üzerinde olmalarını gerektirir: y = x.
2. **r sistematik sapmayı görmez.** AI her hastaya sabit 30 mL ekleseydi r hiç değişmezdi; oysa her sonuç yanlış olurdu.
3. **r verinin aralığına bağlıdır.** Çok küçük ve çok büyük organları eklemek r'yi şişirir, her hastadaki uyumsuzluk aynı kalsa bile.

Aşağıdaki üç gösterim bu noktaları çalışma verisi üzerinde gösteriyor.

### 3.1 The usual scatter plot / Alışılmış dağılım grafiği

**EN:** AI against reference, with the identity line y = x. Points close to the line indicate agreement; a high r alone does not.

**TR:** AI'a karşı referans, özdeşlik doğrusu y = x ile birlikte. Noktaların doğruya yakın olması uyumu gösterir; yüksek r tek başına göstermez.

In [ ]:
cut = cfg["cutoff"]["value"]
r_p = stats.pearsonr(ref, ai)[0]
fig, ax = plt.subplots(figsize=(7, 7))
lo, hi = min(ai.min(), ref.min()), max(ai.max(), ref.max())
ax.plot([lo, hi], [lo, hi], color=COL["grey"], lw=1, ls="--", label="Identity line (y = x)")
ax.scatter(ref, ai, s=26, color=COL["ink"], alpha=0.75)
ax.axvline(cut, color=COL["accent"], lw=1, ls=":")
ax.axhline(cut, color=COL["accent"], lw=1, ls=":", label=f"Clinical cut-off {cut:g} {U}")
ax.set_title(f"{cfg['title']}\nPearson r = {r_p:.3f}", loc="left", fontsize=11)
ax.set_xlabel(f"Reference: mean of 3 experts ({U})"); ax.set_ylabel(f"AI ({U})")
ax.set_aspect("equal"); ax.grid(alpha=0.25); ax.legend(fontsize=9, frameon=False)
fig.tight_layout(); save(fig, "02_scatter_ai_vs_reference.png"); plt.show()

### 3.2 Demonstration: a biased AI keeps the same r / Gösterim: sapmalı bir AI aynı r'yi korur

**EN:** Two hypothetical faulty AIs are created from the real AI output:
- **AI + constant:** adds a fixed amount to every patient (a calibration error).
- **AI × 1.20:** reads every organ 20 % larger (a scaling error).

Compare r with the Bland-Altman results. r stays exactly the same, while the bias and the LoA reveal the error immediately.

**TR:** Gerçek AI çıktısından iki varsayımsal hatalı AI üretilir:
- **AI + sabit:** her hastaya sabit bir miktar ekler (kalibrasyon hatası).
- **AI × 1.20:** her organı %20 büyük okur (ölçekleme hatası).

r'yi Bland-Altman sonuçlarıyla karşılaştır. r birebir aynı kalır; bias ve LoA ise hatayı hemen gösterir.

In [ ]:
shift = 30.0 if U == "mL" else 2.0
variants = {"Original AI": ai, f"AI + {shift:g} {U}": ai + shift, "AI x 1.20": ai * 1.20}
rows = []
for name, v in variants.items():
    rr = bland_altman(differences(v, ref, pr["scale"]))
    rows.append({"Variant": name, "Pearson r": stats.pearsonr(ref, v)[0], "ICC(A,1)": icc_a1(v, ref),
                 f"Bias ({PU})": rr["bias"], f"Lower LoA ({PU})": rr["loa_lo"], f"Upper LoA ({PU})": rr["loa_hi"],
                 "Decision": verdict(acceptance(rr, pr["limit"])).split("|")[0].split(":")[1].strip()})
demo = pd.DataFrame(rows)
display(demo.style.format({"Pearson r": "{:.4f}", "ICC(A,1)": "{:.3f}", f"Bias ({PU})": "{:+.2f}",
                           f"Lower LoA ({PU})": "{:+.2f}", f"Upper LoA ({PU})": "{:+.2f}"}).hide(axis="index"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for ax, (name, v) in zip(axes, variants.items()):
    rr = bland_altman(differences(v, ref, pr["scale"]))
    plot_ba(v, ref, pr["scale"], rr, pr["limit"], f"{name}\nr = {stats.pearsonr(ref, v)[0]:.4f}",
            ax=ax, legend=False)
fig.tight_layout(); save(fig, "02b_correlation_trap.png"); plt.show()

**EN:** Pearson r is exactly the same for all three, because r does not change when a constant is added or when values are multiplied by a constant. Yet the Bland-Altman results move far outside the limit. ICC(A,1) for absolute agreement does drop, which is why ICC is a better supportive measure than r. Even so, ICC gives no answer in clinical units.

**TR:** Pearson r üçünde de birebir aynıdır, çünkü r bir sabit eklendiğinde ya da değerler bir sabitle çarpıldığında değişmez. Oysa Bland-Altman sonuçları ise sınırın çok dışına çıkar. Mutlak uyum için ICC(A,1) düşer; bu yüzden ICC destekleyici ölçü olarak r'den daha iyidir. Yine de ICC klinik birimde bir cevap vermez.

### 3.3 Demonstration: r depends on the range / Gösterim: r aralığa bağlıdır

**EN:** The cases are split into the lower, middle and upper thirds of the reference values. Within each third the disagreement is similar, but r is much lower because the spread of the true values is smaller. The full data set, which spans the whole range, gives the highest r. A study that enrols very different patient sizes can therefore report an impressive r without the agreement being any better.

**TR:** Vakalar referans değerlerinin alt, orta ve üst üçte birine ayrılır. Her dilimde uyumsuzluk benzerdir ama r çok daha düşüktür, çünkü gerçek değerlerin yayılımı daha küçüktür. Tüm aralığı kapsayan tam veri en yüksek r'yi verir. Dolayısıyla çok farklı boyutlarda hastalar alan bir çalışma, uyum hiç iyileşmeden etkileyici bir r raporlayabilir.

In [ ]:
terc = pd.qcut(ref, 3, labels=["Lower third", "Middle third", "Upper third"])
rows = []
for name, m in [("All cases", np.ones(len(ref), bool))] + [(t, (terc == t)) for t in terc.categories]:
    rr = bland_altman(differences(ai[m], ref[m], pr["scale"]))
    rows.append({"Subset": name, "n": int(m.sum()),
                 f"Reference range ({U})": f"{ref[m].min():.1f} - {ref[m].max():.1f}",
                 "Pearson r": stats.pearsonr(ref[m], ai[m])[0],
                 f"SD of differences ({PU})": rr["sd"], f"LoA width ({PU})": rr["loa_hi"] - rr["loa_lo"]})
display(pd.DataFrame(rows).style.format({"Pearson r": "{:.3f}", f"SD of differences ({PU})": "{:.2f}",
                                         f"LoA width ({PU})": "{:.2f}"}).hide(axis="index"))

**Takeaway / Sonuç:**
- **EN:** Report r if you like, but never use it to claim agreement. The claim rests on the bias, the LoA and the pre-specified limit.
- **TR:** r'yi raporlayabilirsin, ama asla uyum iddiası için kullanma. İddia bias, LoA ve önceden belirlenen sınıra dayanır.

## 4. Secondary endpoint and choice of scale / İkincil sonlanım ve ölçek seçimi

**EN:** The primary endpoint uses one scale (absolute or percent). The secondary endpoint repeats the analysis on the other scale. This matters because the two scales answer different clinical questions:

| Scale | Question it answers | When it is the natural choice |
|---|---|---|
| Absolute (mL, cm) | "How many mL or cm can the AI be off?" | The error is roughly constant across sizes; the clinical decision uses a fixed threshold (e.g. PVR 200 mL) |
| Percent (%) | "By what fraction can the AI be off?" | The error grows with size (heteroscedasticity); sizes span a wide range |

Reporting both lets a reader judge the error in the unit they care about. The secondary endpoint here is **descriptive**: it has no pre-specified limit and does not affect the PASS/FAIL decision.

**TR:** Birincil sonlanım tek bir ölçek kullanır (mutlak ya da yüzde). İkincil sonlanım analizi diğer ölçekte tekrarlar. Bu önemlidir, çünkü iki ölçek farklı klinik soruları cevaplar:

| Ölçek | Cevapladığı soru | Ne zaman doğal seçimdir |
|---|---|---|
| Mutlak (mL, cm) | "AI kaç mL ya da cm yanılabilir?" | Hata boyuttan bağımsız yaklaşık sabitse; klinik karar sabit bir eşiğe dayanıyorsa (ör. PVR 200 mL) |
| Yüzde (%) | "AI hangi oranda yanılabilir?" | Hata boyutla büyüyorsa (heteroskedastisite); boyutlar geniş bir aralığa yayılıyorsa |

İkisini birlikte raporlamak, okuyucunun hatayı önemsediği birimde değerlendirmesini sağlar. Buradaki ikincil sonlanım **tanımlayıcıdır**: önceden belirlenmiş sınırı yoktur ve PASS/FAIL kararını etkilemez.

### 4.1 Which scale fits the data? / Veriye hangi ölçek uyuyor?

**EN:** The two panels below show the same patients on both scales. The check is whether the **spread** of the differences changes along the x-axis:

- On the **absolute** scale, a funnel that opens to the right means the error grows with size. The fixed-unit LoA would then be too wide for small organs and too narrow for large ones.
- On the **percent** scale, a funnel that opens to the *left* means the percentage inflates for small values. For example, a 10 mL error on a 20 mL bladder is 50 %.

The Spearman correlation between |difference| and the mean puts a number on this. A clearly positive ρ means the spread grows with size; a clearly negative ρ means it shrinks. The scale with ρ closer to 0 fits better.

**TR:** Aşağıdaki iki panel aynı hastaları iki ölçekte gösterir. Kontrol edilen şey, farkların **yayılımının** x ekseni boyunca değişip değişmediğidir:

- **Mutlak** ölçekte sağa doğru açılan bir huni, hatanın boyutla büyüdüğünü gösterir. O durumda sabit birimli LoA küçük organlar için fazla geniş, büyük organlar için fazla dar olur.
- **Yüzde** ölçekte *sola* doğru açılan bir huni, küçük değerlerde yüzdenin şiştiğini gösterir. Örneğin 20 mL'lik bir mesanede 10 mL hata %50'dir.

|fark| ile ortalama arasındaki Spearman korelasyonu bunu sayıya döker. Belirgin pozitif ρ yayılımın boyutla büyüdüğünü, belirgin negatif ρ küçüldüğünü gösterir. ρ'su 0'a daha yakın olan ölçek veriye daha iyi uyar.

In [ ]:
mean_all = (ai + ref) / 2
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
rows = []
for ax, sc in zip(axes, ["abs", "pct"]):
    dd = differences(ai, ref, sc)
    rr = bland_altman(dd)
    rho, p = stats.spearmanr(mean_all, np.abs(dd))
    rows.append({"Scale": "Absolute" if sc == "abs" else "Percent",
                 "Unit": scale_unit(sc), "Spearman rho (|diff| vs mean)": rho, "p": p,
                 "Bias": rr["bias"], "Lower LoA": rr["loa_lo"], "Upper LoA": rr["loa_hi"],
                 "Used as": "primary" if sc == pr["scale"] else "secondary"})
    lim = pr["limit"] if sc == pr["scale"] else None
    plot_ba(ai, ref, sc, rr, lim,
            f"{'Absolute' if sc == 'abs' else 'Percent'} scale (all {rr['n']} cases)\n"
            f"Spearman rho(|diff|, mean) = {rho:+.2f} (p = {p:.3f})", ax=ax, legend=False)
fig.tight_layout(); save(fig, "03a_scale_comparison.png"); plt.show()
display(pd.DataFrame(rows).style.format({"Spearman rho (|diff| vs mean)": "{:+.2f}", "p": "{:.3f}",
                                         "Bias": "{:+.2f}", "Lower LoA": "{:+.2f}",
                                         "Upper LoA": "{:+.2f}"}).hide(axis="index"))

In [ ]:
rho_abs, rho_pct = rows[0]["Spearman rho (|diff| vs mean)"], rows[1]["Spearman rho (|diff| vs mean)"]
better = "Absolute" if abs(rho_abs) < abs(rho_pct) else "Percent"
better_tr = "Mutlak" if better == "Absolute" else "Yüzde"
display(Markdown(
f"**EN:** Spearman ρ is {rho_abs:+.2f} on the absolute scale and {rho_pct:+.2f} on the percent scale. "
f"On this data the spread is more stable on the **{better.lower()}** scale.\n\n"
f"**TR:** Spearman ρ mutlak ölçekte {rho_abs:+.2f}, yüzde ölçekte {rho_pct:+.2f}. "
f"Bu veride yayılım **{better_tr.lower()}** ölçekte daha sabit."))

**EN: How the scales were chosen in these two studies**

- **Bladder:** the absolute differences clearly fan out as the volume grows, and the percent differences inflate below about 100 mL. Neither scale is perfect. The primary endpoint stays in mL for two reasons: the clinical decision (PVR > 200 mL) is made in mL, and FDA K230497 reported mL. The price is that a single ±mL interval is too generous for small bladders and too strict for large ones. The secondary percent analysis, restricted to reference ≥ 100 mL where percentages behave well, covers that gap. This trade-off should be stated openly in a protocol.
- **Liver:** the absolute error grows mildly with organ length, and the percent scale removes this almost completely. So the primary endpoint uses percent, and the secondary endpoint reports cm for clinicians who think in centimetres.

The general rule: choose the scale on which the spread is stable, unless a clinical threshold or a regulatory precedent requires a specific unit; in that case, report the other scale as a secondary endpoint and explain the choice.

**TR: Bu iki çalışmada ölçekler nasıl seçildi**

- **Mesane:** mutlak farklar hacim büyüdükçe belirgin şekilde yelpaze gibi açılıyor; yüzde farklar ise yaklaşık 100 mL'nin altında şişiyor. İki ölçek de kusursuz değil. Birincil sonlanım iki nedenle mL'de kalıyor: klinik karar (PVR > 200 mL) mL ile veriliyor ve FDA K230497 mL raporladı. Bedeli şu: tek bir ±mL aralığı küçük mesaneler için fazla cömert, büyük mesaneler için fazla katı. Yüzdelerin düzgün davrandığı referans ≥ 100 mL vakalarla sınırlı ikincil yüzde analizi bu boşluğu kapatıyor. Bu ödünleşim protokolde açıkça yazılmalıdır.
- **Karaciğer:** mutlak hata organ uzunluğuyla hafifçe büyüyor; yüzde ölçek bunu neredeyse tamamen ortadan kaldırıyor. Bu yüzden birincil sonlanım yüzde kullanıyor, ikincil sonlanım ise santimetreyle düşünen klinisyenler için cm raporluyor.

Genel kural: yayılımın sabit olduğu ölçeği seç. Bir klinik eşik ya da düzenleyici emsal belirli bir birim gerektiriyorsa onu kullan; o durumda diğer ölçeği ikincil sonlanım olarak raporla ve seçimi gerekçelendir.

### 4.2 Secondary Bland-Altman / İkincil Bland-Altman

**EN:** The same five steps as in section 2, on the secondary scale. Read the table and the plot the same way. Because no limit was pre-specified, the result is reported but not judged.

**TR:** 2. bölümdeki beş adımın aynısı, ikincil ölçekte. Tabloyu ve grafiği aynı şekilde oku. Önceden sınır belirlenmediği için sonuç raporlanır ama karara bağlanmaz.

In [ ]:
se = cfg["secondary"]
SU = scale_unit(se["scale"])
mask = np.ones(len(df), bool) if not se.get("min_ref") else ref >= se["min_ref"]
d2 = differences(ai[mask], ref[mask], se["scale"])
r2 = bland_altman(d2)
a2 = acceptance(r2, se["limit"])
pb2 = proportional_bias(ai[mask], ref[mask], d2)

print(f"Secondary endpoint: {se['name']}")
if se.get("min_ref"):
    print(f"Restricted to reference >= {se['min_ref']:g} {U}: {mask.sum()} of {len(mask)} cases")
display(res_table(r2))
print("DECISION:", verdict(a2))
print(f"Proportional error: slope = {pb2['slope']:+.4f}, p = {pb2['p']:.4f}")
plot_ba(ai[mask], ref[mask], se["scale"], r2, se["limit"],
        f"{cfg['title']}: secondary, {se['name']} (n={r2['n']})", "03_bland_altman_secondary.png")

In [ ]:
display(Markdown(
f"**EN:** On the secondary scale the bias is {r2['bias']:+.2f} {SU} and the LoA are {r2['loa_lo']:+.2f} to {r2['loa_hi']:+.2f} {SU}. "
f"So, for about 95 % of patients{' with reference ≥ ' + format(se['min_ref'], 'g') + ' ' + U if se.get('min_ref') else ''}, "
f"the AI differs from the experts by between {r2['loa_lo']:+.2f} and {r2['loa_hi']:+.2f} {SU}.\n\n"
f"**TR:** İkincil ölçekte bias {r2['bias']:+.2f} {SU}, LoA {r2['loa_lo']:+.2f} ile {r2['loa_hi']:+.2f} {SU}. "
f"Yani {('referansı ≥ ' + format(se['min_ref'], 'g') + ' ' + U + ' olan ') if se.get('min_ref') else ''}hastaların yaklaşık %95'inde "
f"AI uzmanlardan {r2['loa_lo']:+.2f} ile {r2['loa_hi']:+.2f} {SU} arasında farklı ölçüyor."))

## 5. ICC, MAE and clinical cut-off / ICC, MAE ve klinik eşik

**EN:** Supportive measures that FDA summaries often report next to Bland-Altman. ICC(A,1) is an absolute-agreement index between 0 and 1. MAE is the mean absolute error in the measurement unit. The clinical cut-off analysis asks whether the AI puts the patient on the same side of the decision threshold as the experts. The PASS/FAIL decision is still made with the LoA.

**TR:** FDA özetlerinde Bland-Altman'ın yanında sık raporlanan destekleyici ölçüler. ICC(A,1), 0 ile 1 arasında bir mutlak uyum indeksidir. MAE ölçüm birimindeki ortalama mutlak hatadır. Klinik eşik analizi, AI'ın hastayı karar eşiğinin uzmanlarla aynı tarafına koyup koymadığını sorar. PASS/FAIL kararı yine LoA ile verilir.

In [ ]:
icc = icc_a1(ai, ref)
mae = np.mean(np.abs(ai - ref))
cat = categorical(ai, ref, cut)
display(pd.DataFrame([{"ICC(A,1)": icc, "Pearson r": r_p, f"MAE ({U})": mae}]))

print(cfg["cutoff"]["label"])
display(pd.DataFrame([[cat["tp"], cat["fn"]], [cat["fp"], cat["tn"]]],
                     index=["Reference positive", "Reference negative"], columns=["AI positive", "AI negative"]))
display(pd.DataFrame([
    {"Metric": "Sensitivity", "Value": cat["sens"][0], "95% CI (Wilson)": f"{cat['sens'][1]:.3f} - {cat['sens'][2]:.3f}"},
    {"Metric": "Specificity", "Value": cat["spec"][0], "95% CI (Wilson)": f"{cat['spec'][1]:.3f} - {cat['spec'][2]:.3f}"},
    {"Metric": "Overall agreement", "Value": cat["agreement"], "95% CI (Wilson)": ""},
    {"Metric": "Cohen kappa", "Value": cat["kappa"], "95% CI (Wilson)": ""},
]).style.format({"Value": "{:.3f}"}).hide(axis="index"))

## 6. AI vs. human readers / AI ve insan okuyucular

**EN:** The AI's LoA is compared with how much the readers disagree with each other. If the AI's interval is no wider than the readers', the AI disagrees with the experts no more than the experts disagree among themselves. This also shows whether the acceptance limit is realistic.

**TR:** AI'ın LoA'sı, okuyucuların birbirinden ne kadar farklı ölçtüğüyle kıyaslanır. AI'ın aralığı okuyucularınkinden geniş değilse, AI uzmanlardan uzmanların birbirinden farklılaştığından daha fazla farklılaşmıyor demektir. Bu karşılaştırma kabul sınırının gerçekçi olup olmadığını da gösterir.

In [ ]:
rd = cfg["readers"]; bench = []
for i in range(len(rd)):
    for j in range(i + 1, len(rd)):
        rr = bland_altman(differences(df[rd[i]], df[rd[j]], pr["scale"]))
        bench.append({"label": f"Reader {i+1} - Reader {j+1}", **rr, "is_ai": False})
for i in range(len(rd)):
    others = df[[c for k, c in enumerate(rd) if k != i]].mean(axis=1)
    rr = bland_altman(differences(df[rd[i]], others, pr["scale"]))
    bench.append({"label": f"Reader {i+1} - mean of other readers", **rr, "is_ai": False})
bench.append({"label": "AI - mean of 3 readers", **r1, "is_ai": True})
bench = pd.DataFrame(bench)
bench["LoA width"] = bench["loa_hi"] - bench["loa_lo"]
display(bench[["label", "n", "bias", "loa_lo", "loa_hi", "LoA width"]])
plot_intervals(bench, pr["limit"], "AI vs. readers: bias (dot) and 95% LoA (line)",
               ylabel_for(pr["scale"]), "04_ai_vs_human_loa.png")

## 7. Subgroups / Alt gruplar

**EN:** Exploratory: subgroups do not change the study decision, but FDA expects consistent performance across site, device, sex, BMI and age. Subgroups whose LoA exceeds the limit are flagged. Small subgroups give wide, unstable LoA, so read them with care.

**TR:** Keşifseldir: alt gruplar çalışma kararını değiştirmez, ama FDA merkez, cihaz, cinsiyet, BMI ve yaş genelinde tutarlı performans görmek ister. LoA'sı sınırı aşan alt gruplar işaretlenir. Küçük alt gruplar geniş ve kararsız LoA verir, dikkatli yorumlanmalıdır.

In [ ]:
SUBGROUP_COLS = ["site", "device_type", "sex", "bmi_category", "age_group", "image_quality"]
rows = []
for col in SUBGROUP_COLS:
    for lvl, g in df.groupby(col):
        if len(g) >= 5:
            gr = bland_altman(differences(g[cfg["ai"]], g["reference"], pr["scale"]))
            flag = pr["limit"] is not None and (gr["loa_lo"] < -pr["limit"] or gr["loa_hi"] > pr["limit"])
            rows.append({"label": f"{col}: {lvl}", **gr, "Exceeds limit": "yes" if flag else ""})
subs = pd.DataFrame(rows)
display(subs[["label", "n", "bias", "loa_lo", "loa_hi", "Exceeds limit"]])
plot_intervals(subs, pr["limit"], "Subgroups: bias and 95% LoA (exploratory)",
               ylabel_for(pr["scale"]), "05_subgroup_forest.png")

## 8. Sample size planning / Örneklem büyüklüğü planlama

**EN:** A simulation estimates, for each n, the probability that both LoA confidence intervals fall inside the limit (the strict criterion), given an assumed bias and SD. In a real study the SD comes from pilot data, and 10–20 % is added for non-analyzable cases.

**TR:** Simülasyon, varsayılan bias ve SD ile her n için her iki LoA güven aralığının sınır içinde kalma olasılığını (sıkı kriter) tahmin eder. Gerçek bir çalışmada SD pilot veriden alınır ve analiz edilemeyecek vakalar için %10–20 eklenir.

In [ ]:
def simulate_power(bias, sd, delta, n, nsim=1500, rng=None):
    rng = rng or np.random.default_rng(1); ok = 0
    for _ in range(nsim):
        r = bland_altman(rng.normal(bias, sd, n))
        ok += r["loa_lo_ci_lo"] >= -delta and r["loa_hi_ci_hi"] <= delta
    return ok / nsim

if RUN_POWER:
    p = cfg["power"]; rng = np.random.default_rng(7)
    ns = np.arange(20, 401, 20)
    pw = [simulate_power(p["bias"], p["sd"], p["delta"], n, 1500, rng) for n in ns]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(ns, pw, "o-", color=COL["ink"])
    ax.axhline(0.8, color=COL["accent"], ls="--", lw=1.2, label="80% power")
    ax.axhline(0.9, color=COL["blue"], ls="--", lw=1.2, label="90% power")
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Analyzable cases (n)"); ax.set_ylabel("P(both LoA CIs within limit)")
    ax.set_title(f"Sample size: bias={p['bias']}, SD={p['sd']}, limit=±{p['delta']}", loc="left", fontsize=11)
    ax.grid(alpha=0.25); ax.legend(frameon=False)
    fig.tight_layout(); save(fig, "06_sample_size_power.png"); plt.show()
    for target in (0.8, 0.9):
        n_ok = next((n for n, v in zip(ns, pw) if v >= target), None)
        print(f"{int(target*100)}% power -> minimum n: " + (str(n_ok) if n_ok else "not reached up to 400"))
else:
    print("Skipped (RUN_POWER = False)")

## 9. Save outputs / Çıktıları kaydet

In [ ]:
if SAVE_OUTPUTS:
    ep = pd.DataFrame([
        {"endpoint": "primary: " + pr["name"], "limit": pr["limit"], **r1,
         "pass_point": a1["point"], "pass_strict": a1["strict"]},
        {"endpoint": "secondary: " + se["name"], "limit": se["limit"], **r2,
         "pass_point": a2["point"], "pass_strict": a2["strict"]}])
    ep.round(4).to_csv(os.path.join(OUT, "endpoints.csv"), index=False)
    bench.round(4).to_csv(os.path.join(OUT, "reader_benchmark.csv"), index=False)
    subs.round(4).to_csv(os.path.join(OUT, "subgroups.csv"), index=False)
    print("Saved to:", os.path.abspath(OUT))
    for f in sorted(os.listdir(OUT)):
        print("  -", f)

print(f"\nSUMMARY - {cfg['title']}")
print(f"Analyzed {r1['n']} of {len(raw)} cases")
print(f"Bias {r1['bias']:+.2f} {PU}; LoA {r1['loa_lo']:+.2f} / {r1['loa_hi']:+.2f} {PU}; limit ±{pr['limit']:g} {PU}")
print(verdict(a1))

## 10. How to run / Nasıl çalıştırılır

**EN:** Run these commands in a terminal, in the folder that holds this notebook. The CSV files are optional; if they are missing, the notebook generates them.

**TR:** Bu komutları notebook'un bulunduğu klasörde, terminalde çalıştır. CSV dosyaları isteğe bağlı; yoksa notebook kendisi üretir.

```bash
# 1) Install packages
pip install numpy pandas scipy matplotlib jupyter

# 2) Open interactively, then Kernel -> Restart & Run All
jupyter notebook bland_altman_study.ipynb

# 3) Run headless and keep the executed copy
jupyter nbconvert --to notebook --execute bland_altman_study.ipynb --output bladder_executed.ipynb

# 4) HTML report
jupyter nbconvert --to html --execute bland_altman_study.ipynb
```

**EN:** For the liver study, set `STUDY = "liver"` in section 0 and run all cells again. To use your own data, prepare a CSV with the same column names and set `CSV_PATH`. The next cell runs both studies in one go.

**TR:** Karaciğer çalışması için 0. bölümde `STUDY = "liver"` yap ve tüm hücreleri yeniden çalıştır. Kendi verini kullanmak için aynı sütun adlarıyla bir CSV hazırla ve `CSV_PATH`'i ayarla. Sonraki hücre iki çalışmayı tek seferde çalıştırır.

In [ ]:
# Run both studies headless and save executed copies.
# Uncomment to use. Each run executes this notebook with a different STUDY value.

# import nbformat
# from nbclient import NotebookClient
# for study in ["bladder", "liver"]:
#     nb = nbformat.read("bland_altman_study.ipynb", as_version=4)
#     settings = next(c for c in nb.cells if c.cell_type == "code" and 'STUDY = "bladder"' in c.source)
#     settings.source = settings.source.replace('STUDY = "bladder"', f'STUDY = "{study}"')
#     nb.cells = nb.cells[:-1]                      # drop this cell to avoid recursion
#     NotebookClient(nb, timeout=900, kernel_name="python3").execute()
#     nbformat.write(nb, f"{study}_executed.ipynb")
#     print(f"done: {study}_executed.ipynb -> results_{study}/")